# 03 - Feature engineering
Turns every clip into features and saves them to features/:
- Log-mel spectrogram (64 x 259 image per clip), normalized with train statistics - input for the CNN
- MFCC means (13 numbers per clip) - input for simple models

Before running: run 01_data_preparation once, so data_cache/ exists.

what do you want for the models?
how many features?
4-8 simple for the classic ml?

CNN makes it own features? need to give it like 
- log-mel-spectrum
- fast fourier transom
- mfcc?
- smt?

In [1]:
import os
import numpy as np
import librosa

# load the data saved by 01_data_preparation
X_train = np.load("data_cache/X_train.npy")
y_train = np.load("data_cache/y_train.npy")
X_val = np.load("data_cache/X_val.npy")
y_val = np.load("data_cache/y_val.npy")
val_recordings = np.load("data_cache/val_recordings.npy")

print("Train:", X_train.shape, "Val:", X_val.shape)

Train: (357, 132300) Val: (442, 132300)


## 1. Log-mel spectrograms

describes image of the sound
representing like energy over time on a logarthmic db scale.
https://librosa.org/doc/latest/api/generated/librosa.feature.melspectrogram.html


In [2]:
# Log-mel spectrogram: input for the CNN
# using librosa to compute the log-mel spectrograms.
def log_mel(x):
    mel = librosa.feature.melspectrogram(y=x, sr=44100, n_fft=2048, hop_length=512, n_mels=64)
    return librosa.power_to_db(mel)                     


log_M_train = np.array([log_mel(x) for x in X_train])
log_M_val = np.array([log_mel(x) for x in X_val])

In [3]:
# normalize log mel spectrograms
# cnn input for normalized log-mel image.

mean = log_M_train.mean() # average of all values
std = log_M_train.std() # average deviation of all values
log_M_train = (log_M_train - mean) / std # centers data around 0 and scales it
log_M_val = (log_M_val - mean) / std

## 2. MFCC

makes like 13 different numbers per clip
change n_mfcc = 13 to get more or fewer features out of it.
https://librosa.org/doc/latest/api/generated/librosa.feature.mfcc.html#librosa.feature.mfcc

- MFCC 0 should describe overall loudness
- MFCc 1 balance between low and high hz
- MFCC 2-12 other stuff like humps, small peaks, dips etc

In [4]:
# MFCC: n_mfcc is features that describe the shape of the spectrum.
# 
def mfcc_means(x):
    mfcc = librosa.feature.mfcc(y=x, sr=44100, n_mfcc=13, n_fft=2048, hop_length=512)
    return mfcc.mean(axis=1)

F_train = np.array([mfcc_means(x) for x in X_train])
F_val = np.array([mfcc_means(x) for x in X_val])

print("MFCC train:", F_train.shape, "val:", F_val.shape)

MFCC train: (357, 13) val: (442, 13)


## 3. Save features for the modellers

In [5]:
os.makedirs("features", exist_ok=True)

np.save("features/log_M_train.npy", log_M_train)
np.save("features/log_M_val.npy", log_M_val)
np.save("features/F_train.npy", F_train)
np.save("features/F_val.npy", F_val)
np.save("features/y_train.npy", y_train)
np.save("features/y_val.npy", y_val)
np.save("features/val_recordings.npy", val_recordings)
np.save("features/norm_stats.npy", np.array([mean, std]))   # train mean and std, needed for test

print("Saved features to features/")

Saved features to features/


## 4. Test features

Uses the train mean and std, so test is processed exactly like the training data.

Requires LOAD_TEST = True in the last cell of 01_data_preparation first.

In [6]:
# Set to True ONLY at the very end, when the models are finished.
LOAD_TEST = False

if LOAD_TEST:
    X_test = np.load("data_cache/X_test.npy")
    y_test = np.load("data_cache/y_test.npy")
    test_recordings = np.load("data_cache/test_recordings.npy")

    log_M_test = np.array([log_mel(x) for x in X_test])
    log_M_test = (log_M_test - mean) / std          # train statistics, nothing calculated from test
    F_test = np.array([mfcc_means(x) for x in X_test])

    np.save("features/log_M_test.npy", log_M_test)
    np.save("features/F_test.npy", F_test)
    np.save("features/y_test.npy", y_test)
    np.save("features/test_recordings.npy", test_recordings)

    print("Saved test features:", log_M_test.shape, F_test.shape)
else:
    print("Test features NOT made (LOAD_TEST = False)")

Test features NOT made (LOAD_TEST = False)
